In [ ]:
"""
PINN — 二维 Poisson 方程求解（重构版）
========================================
PDE:   -Δu(x,y) = f(x,y),   (x,y) ∈ Ω = [-1,1]²
边界:   u = g_b(x,y),        (x,y) ∈ ∂Ω   (Dirichlet)

精确解: F(x,y) = g(x) + g(y),  g(x) = exp(-x²) sin(30x²)
源项:   f(x,y) = -(g''(x) + g''(y))     ← 注意负号，匹配 -Δu = f
边界值: g_b(x,y) = F(x,y) 在 ∂Ω

损失函数:
    L = λ_r · MSE[ -(u_xx + u_yy) - f ]  +  λ_b · MSE[ u - g_b ]

修复说明（对比原始版本）：
  1. [Bug 修复] g_second 推导有误：cos 项系数应为 2μ - 8μx²，原来写成 2μ - 4μx²
  2. [Bug 修复] laplacian 内部不再 detach+重设 requires_grad；
               改为在训练循环中 clone().requires_grad_(True)，计算图干净
  3. [改进] 内部采样改用网格+随机混合，更均匀覆盖高频振荡区域
  4. [改进] 加回 CosineAnnealingLR，帮助训练后期收敛
  5. [改进] 增加可视化输出，方便核验结果

网络:  2 → [hidden]*num_layers → 1，tanh 激活
训练:  15000 epoch, Adam(lr=1e-3) + CosineAnnealingLR
保存:  权重文件 + 训练曲线 (res_loss, bc_loss, total_loss, l2_error)
"""

import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
torch.manual_seed(42)
np.random.seed(42)

CHECKPOINT_DIR = "checkpoints_pinn_poisson"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

MU       = 30.0
LAMBDA_R = 1.0
LAMBDA_B = 10.0


# ==============================================================================
# 精确解 & 源项
# ==============================================================================

def g_func(x: torch.Tensor) -> torch.Tensor:
    """g(x) = exp(-x²) sin(μx²)"""
    return torch.exp(-x ** 2) * torch.sin(MU * x ** 2)


def g_second(x: torch.Tensor) -> torch.Tensor:
    """
    g(x)  = e^{-x²} sin(μx²)
    g'(x) = e^{-x²} [(-2x) sin(μx²) + (2μx) cos(μx²)]
           = e^{-x²} · 2x · [-sin(μx²) + μ cos(μx²)]

    g''(x) = 对 g'(x) 再求导，令 s = sin(μx²), c = cos(μx²)：

    g'  = e^{-x²} · 2x · (-s + μc)
    g'' = -2x · e^{-x²} · 2x · (-s + μc)          # 来自 e^{-x²} 的导数
        + e^{-x²} · 2 · (-s + μc)                  # 来自 2x 的导数
        + e^{-x²} · 2x · (-2μx·c - 2μ²x·s)        # 来自 (-s + μc) 的导数 (链式)

    整理（提取 e^{-x²}）：
      = e^{-x²} [
          (-4x² + 2)·(-s + μc)          # 合并前两项（-4x²·(·) + 2·(·)）
          + 2x · (-2μx·c - 2μ²x·s)      # 第三项展开
        ]
      = e^{-x²} [
          (4x² - 4μ²x² - 2) s            # sin 项
          + (2μ - 8μx²)      c            # cos 项   ← 原代码写成 2μ - 4μx²，有误！
        ]

    【修复】cos 项系数：
      原代码：2*MU - 4*MU*x**2
      正确值：2*MU - 8*MU*x**2
      差别来自第三项展开后与前两项合并时遗漏了 -4μx² 贡献。
    """
    ex      = torch.exp(-x ** 2)
    s       = torch.sin(MU * x ** 2)
    c       = torch.cos(MU * x ** 2)
    coeff_s = 4 * x**2 - 4 * MU**2 * x**2 - 2   # sin 项系数（原代码正确）
    coeff_c = 2 * MU - 8 * MU * x**2             # cos 项系数（原代码有 bug，此处修复）
    return ex * (coeff_s * s + coeff_c * c)


def u_exact(xy: torch.Tensor) -> torch.Tensor:
    """精确解 F(x,y) = g(x) + g(y)；xy: (N,2) → (N,1)"""
    return g_func(xy[:, 0:1]) + g_func(xy[:, 1:2])


def f_source(xy: torch.Tensor) -> torch.Tensor:
    """
    源项 f，满足 -Δu = f
    Δu = g''(x) + g''(y)  →  f = -(g''(x) + g''(y))
    """
    return -(g_second(xy[:, 0:1]) + g_second(xy[:, 1:2]))


def g_boundary(xy: torch.Tensor) -> torch.Tensor:
    return u_exact(xy)


# ==============================================================================
# 网络
# ==============================================================================

class PINN(nn.Module):
    def __init__(self, hidden_dim: int = 128, num_layers: int = 6):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(num_layers - 2):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)


# ==============================================================================
# 自动微分：计算 Δu = u_xx + u_yy
#
# 【修复】原代码在 laplacian 内部 detach().requires_grad_(True)，
#   会在临时局部张量上积累梯度，计算图与 model.parameters() 的连接不干净。
#   正确做法：在训练循环外 clone().requires_grad_(True)，
#   laplacian 函数直接使用已带梯度的张量，不再内部 detach。
# ==============================================================================

def laplacian(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    """
    xy: (N,2)，调用前已设置 requires_grad=True（由训练循环负责）
    返回: Δu = u_xx + u_yy，shape (N,1)
    """
    u = model(xy)   # (N,1)

    grad_u = torch.autograd.grad(
        u, xy,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]             # (N,2)

    u_xx = torch.autograd.grad(
        grad_u[:, 0:1], xy,
        grad_outputs=torch.ones_like(grad_u[:, 0:1]),
        create_graph=True,
        retain_graph=True,
    )[0][:, 0:1]    # (N,1)

    u_yy = torch.autograd.grad(
        grad_u[:, 1:2], xy,
        grad_outputs=torch.ones_like(grad_u[:, 1:2]),
        create_graph=True,
        retain_graph=True,
    )[0][:, 1:2]    # (N,1)

    return u_xx + u_yy


# ==============================================================================
# 采样
# ==============================================================================

def sample_interior(n: int, seed: int = 1) -> torch.Tensor:
    """
    [-1,1]² 内部混合采样（均匀网格 + 随机），更好地覆盖高频振荡区域。
    返回普通张量（不设 requires_grad，由训练循环 clone 后设置）。
    """
    rng  = np.random.RandomState(seed)
    side = int(np.sqrt(n // 2))
    gx   = np.linspace(-1 + 1e-4, 1 - 1e-4, side)
    gy   = np.linspace(-1 + 1e-4, 1 - 1e-4, side)
    GX, GY = np.meshgrid(gx, gy)
    xy_grid = np.stack([GX.ravel(), GY.ravel()], axis=1)
    n_rand  = n - xy_grid.shape[0]
    xy_rand = rng.uniform(-1.0, 1.0, size=(n_rand, 2))
    pts     = np.concatenate([xy_grid, xy_rand], axis=0)
    return torch.tensor(pts, dtype=torch.float64, device=device)


def sample_boundary(n_per_edge: int, seed: int = 2) -> torch.Tensor:
    """四条边各 n_per_edge 个点，返回 (4*n_per_edge, 2)"""
    rng    = np.random.RandomState(seed)
    t      = rng.uniform(-1.0, 1.0, n_per_edge)
    bottom = np.stack([t,  -np.ones(n_per_edge)], axis=1)
    top    = np.stack([t,   np.ones(n_per_edge)], axis=1)
    left   = np.stack([-np.ones(n_per_edge), t],  axis=1)
    right  = np.stack([ np.ones(n_per_edge), t],  axis=1)
    pts    = np.concatenate([bottom, top, left, right], axis=0)
    return torch.tensor(pts, dtype=torch.float64, device=device)


# ==============================================================================
# 训练
# ==============================================================================

def train_pinn(
    hidden_dim   : int   = 128,
    num_layers   : int   = 6,
    n_interior   : int   = 10000,
    n_per_edge   : int   = 250,
    total_epochs : int   = 15000,
    lr           : float = 1e-3,
    log_every    : int   = 1,
    lambda_r     : float = LAMBDA_R,
    lambda_b     : float = LAMBDA_B,
):
    # ── 采样 ──────────────────────────────────────────────────────────────────
    xy_int = sample_interior(n_interior, seed=1)   # 普通张量
    f_int  = f_source(xy_int).detach()             # 源项，固定

    xy_bc  = sample_boundary(n_per_edge, seed=2)
    u_bc   = g_boundary(xy_bc).detach()

    print(f"内部配点: {xy_int.shape[0]},  边界配点: {xy_bc.shape[0]}")

    # ── 测试集（100×100 均匀网格）────────────────────────────────────────────
    nx      = 100
    xv, yv  = np.meshgrid(np.linspace(-1, 1, nx), np.linspace(-1, 1, nx))
    xy_test = torch.tensor(
        np.stack([xv.ravel(), yv.ravel()], axis=1),
        dtype=torch.float64, device=device
    )
    u_test         = u_exact(xy_test).detach()
    u_test_sq_mean = torch.mean(u_test ** 2).item()

    # ── 模型 & 优化器 ─────────────────────────────────────────────────────────
    model        = PINN(hidden_dim, num_layers).double().to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"网络参数量: {total_params:,}")

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    # sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, total_epochs, eta_min=1e-5)

    # ── 记录 ──────────────────────────────────────────────────────────────────
    epochs_record, res_losses, bc_losses, total_losses, l2_errors = [], [], [], [], []
    t0 = time.time()

    print(f"\n{'='*60}")
    print(f"  PINN 训练 (重构版)  |  网络: {hidden_dim}×{num_layers}层 tanh")
    print(f"  内部点: {n_interior}, 边界点: {xy_bc.shape[0]}")
    print(f"  损失权重: λ_r={lambda_r}, λ_b={lambda_b}")
    print(f"{'='*60}\n")

    for ep in range(1, total_epochs + 1):
        model.train()
        opt.zero_grad()

        # ── PDE 残差：-Δu - f = 0 ────────────────────────────────────────────
        # 【修复】在训练循环内 clone + requires_grad_(True)，不在 laplacian 内部 detach
        xy_int_r = xy_int.clone().requires_grad_(True)
        lap_u    = laplacian(model, xy_int_r)        # Δu，(N,1)
        loss_r   = torch.mean((-lap_u - f_int) ** 2) # 残差：-Δu - f

        # ── 边界损失 ─────────────────────────────────────────────────────────
        loss_b = nn.functional.mse_loss(model(xy_bc), u_bc)

        loss = lambda_r * loss_r + lambda_b * loss_b
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        # sch.step()

        # ── 记录 & 打印 ───────────────────────────────────────────────────────
        if ep % log_every == 0 or ep == 1:
            with torch.no_grad():
                pred_test = model(xy_test)
                l2 = torch.sqrt(
                    torch.mean((pred_test - u_test) ** 2) / u_test_sq_mean
                ).item()

            epochs_record.append(ep)
            res_losses.append(loss_r.item())
            bc_losses.append(loss_b.item())
            total_losses.append(loss.item())
            l2_errors.append(l2)

            if ep % (log_every * 5) == 0 or ep == 1:
                print(
                    f"ep {ep:6d} | res: {loss_r.item():.3e} | "
                    f"bc: {loss_b.item():.3e} | total: {loss.item():.3e} | "
                    f"L2: {l2:.6f} | {time.time()-t0:.1f}s"
                )

    # ── 最终指标 ──────────────────────────────────────────────────────────────
    with torch.no_grad():
        fp   = model(xy_test)
        fl2  = torch.sqrt(torch.mean((fp - u_test) ** 2) / u_test_sq_mean).item()
        fmae = torch.mean(torch.abs(fp - u_test)).item()
    print(f"\n训练完成 | L2: {fl2:.6f} | MAE: {fmae:.6f} | 时间: {time.time()-t0:.1f}s")

    # ── 保存权重 & 曲线 ───────────────────────────────────────────────────────
    torch.save(
        model.state_dict(),
        os.path.join(CHECKPOINT_DIR, "pinn_poisson_weights.pt")
    )
    np.savez(
        os.path.join(CHECKPOINT_DIR, "pinn_poisson_curves.npz"),
        epochs     = np.array(epochs_record),
        res_loss   = np.array(res_losses),
        bc_loss    = np.array(bc_losses),
        total_loss = np.array(total_losses),
        l2_error   = np.array(l2_errors),
    )
    print(f"权重与曲线已保存至 {CHECKPOINT_DIR}/")

    return dict(
        model=model,
        xy_test=xy_test, u_test=u_test,
        epochs=np.array(epochs_record),
        res_loss=np.array(res_losses),
        bc_loss=np.array(bc_losses),
        total_loss=np.array(total_losses),
        l2_error=np.array(l2_errors),
        fl2=fl2, fmae=fmae,
        hidden_dim=hidden_dim, num_layers=num_layers,
        total_params=total_params,
        n_interior=n_interior, n_bc=xy_bc.shape[0],
    )


# ==============================================================================
# 可视化
# ==============================================================================

def plot_results(res: dict):
    N   = 100
    xyt = res["xy_test"].cpu().numpy()
    X2D = xyt[:, 0].reshape(N, N)
    Y2D = xyt[:, 1].reshape(N, N)

    with torch.no_grad():
        pred_np = res["model"](res["xy_test"]).squeeze(-1).cpu().numpy().reshape(N, N)
    true_np = res["u_test"].squeeze(-1).cpu().numpy().reshape(N, N)
    err_np  = np.abs(pred_np - true_np)

    fig = plt.figure(figsize=(18, 16))
    fig.suptitle(
        "PINN — 2D Poisson Equation  $-\\Delta u = f$\n"
        r"精确解 $F(x,y)=e^{-x^2}\sin(30x^2)+e^{-y^2}\sin(30y^2)$"
        f"\n网络: {res['num_layers']} layers × {res['hidden_dim']} units "
        f"({res['total_params']:,} params) | "
        f"Interior: {res['n_interior']}, BC: {res['n_bc']}\n"
        f"最终相对 L2: {res['fl2']:.6f}  |  MAE: {res['fmae']:.6f}",
        fontsize=12, y=0.995,
    )
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.33)

    vmin, vmax = true_np.min(), true_np.max()

    # 行 0：精确解 / 预测 / 绝对误差
    for col, (data, title, cmap, v) in enumerate(zip(
        [true_np, pred_np, err_np],
        ["Exact $F(x,y)$",
         f"PINN Prediction  (L2={res['fl2']:.4f})",
         "Absolute Error |pred − exact|"],
        ["RdBu_r", "RdBu_r", "Reds"],
        [(vmin, vmax), (vmin, vmax), (None, None)],
    )):
        ax = fig.add_subplot(gs[0, col])
        kw = dict(levels=60, cmap=cmap)
        if v[0] is not None:
            kw.update(vmin=v[0], vmax=v[1])
        cf = ax.contourf(X2D, Y2D, data, **kw)
        plt.colorbar(cf, ax=ax, pad=0.02)
        ax.set_title(title, fontsize=10)
        ax.set_aspect("equal")
        ax.set_xlabel("x"); ax.set_ylabel("y")

    # 行 1：截面比较
    mid  = N // 2
    cuts = [
        (X2D[:, mid], true_np[:, mid], pred_np[:, mid], "Section y=0"),
        (Y2D[mid, :], true_np[mid, :], pred_np[mid, :], "Section x=0"),
        (np.diag(X2D), np.diag(true_np), np.diag(pred_np), "Diagonal x=y"),
    ]
    for col, (xv, yt, yp, title) in enumerate(cuts):
        ax = fig.add_subplot(gs[1, col])
        ax.plot(xv, yt, "#378ADD", lw=2.0, label="Exact")
        ax.plot(xv, yp, "#D85A30", lw=1.5, ls="--", label="PINN")
        ax.set_title(title, fontsize=10)
        ax.legend(fontsize=9); ax.grid(alpha=0.25)

    # 行 2：训练曲线 / 误差分布 / Log 误差热图
    ep = res["epochs"]

    ax = fig.add_subplot(gs[2, 0])
    ax.semilogy(ep, res["total_loss"], "#534AB7", lw=1.8, label="Total")
    ax.semilogy(ep, res["res_loss"],   "#D85A30", lw=1.4, ls="--", label="PDE Residual")
    ax.semilogy(ep, res["bc_loss"],    "#2CA02C", lw=1.4, ls=":",  label="BC")
    ax.set_title("Training Loss (log scale)", fontsize=10)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(fontsize=9); ax.grid(alpha=0.25)

    ax = fig.add_subplot(gs[2, 1])
    ax.semilogy(ep, res["l2_error"], "#534AB7", lw=1.8)
    ax.set_title("Relative L2 Error", fontsize=10)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Rel. L2")
    ax.grid(alpha=0.25)

    ax = fig.add_subplot(gs[2, 2])
    cf = ax.contourf(X2D, Y2D, np.log10(err_np + 1e-12), levels=50, cmap="hot_r")
    plt.colorbar(cf, ax=ax, pad=0.02, label="log₁₀|error|")
    ax.set_title("Log Error Map", fontsize=10)
    ax.set_aspect("equal")
    ax.set_xlabel("x"); ax.set_ylabel("y")

    out_path = "pinn_poisson_fixed_result.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"图像已保存: {out_path}")
    plt.show()


# ==============================================================================
if __name__ == "__main__":
    res = train_pinn()
    plot_results(res)